In [1]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_path = project_root / "data" / "raw" / "Online Retail.xlsx"

retail = pd.read_excel(data_path, engine="openpyxl")

print(f"File: {data_path.relative_to(project_root)}")
print(f"Rows, columns: {retail.shape}")
print("Columns:", retail.columns.tolist())
print("\nData types:\n", retail.dtypes)
print("\nFirst five rows:\n", retail.head())
print("\nMissing values:\n", retail.isna().sum())

File: data\raw\Online Retail.xlsx
Rows, columns: (541909, 8)
Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Data types:
 InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object

First five rows:
   InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     178

In [2]:
print(f"Date range: {retail['InvoiceDate'].min()} to {retail['InvoiceDate'].max()}")
print(f"Distinct invoices: {retail['InvoiceNo'].nunique():,}")
print(f"Distinct stock codes: {retail['StockCode'].nunique():,}")
print(f"Distinct customers (non-missing): {retail['CustomerID'].nunique():,}")
print(f"Countries: {retail['Country'].nunique():,}")
print(f"Rows with negative quantity: {(retail['Quantity'] < 0).sum():,}")
print(f"Rows with zero quantity: {(retail['Quantity'] == 0).sum():,}")
print(f"Rows with negative unit price: {(retail['UnitPrice'] < 0).sum():,}")
print(f"Rows with zero unit price: {(retail['UnitPrice'] == 0).sum():,}")
print("Invoice number examples beginning with C:", retail.loc[retail['InvoiceNo'].astype(str).str.startswith('C'), 'InvoiceNo'].drop_duplicates().head().tolist())

Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00
Distinct invoices: 25,900
Distinct stock codes: 4,070
Distinct customers (non-missing): 4,372
Countries: 38
Rows with negative quantity: 10,624
Rows with zero quantity: 0
Rows with negative unit price: 2
Rows with zero unit price: 2,515


Invoice number examples beginning with C: ['C536379', 'C536383', 'C536391', 'C536506', 'C536543']


# Data understanding

## Initial inspection

The workbook loads successfully as 541,909 invoice-line rows and 8 columns. Dates run from 2010-12-01 to 2011-12-09. It contains 25,900 invoices, 4,070 stock codes, 4,372 non-missing customer IDs, and 38 countries. `CustomerID` is missing on 135,080 rows; `Description` is missing on 1,454. There are 10,624 negative-quantity rows, 2,515 zero-price rows, and 2 negative-price rows. Some invoice numbers begin with `C`.

These values are observations, not cleaning decisions: retain the raw data and establish what the exceptional rows mean before defining sales or demand measures. Each row appears to represent an invoice line, so line count, item quantity, and invoice count are different measures.

## Proposed follow-up analyses (not performed)

The focused transaction-pattern investigation is documented below. The remaining proposals are intentionally deferred.

### 1. Describe baseline demand and time patterns

- **Question:** How does purchased quantity vary across products and over the observed period?
- **Variables:** `StockCode`, `InvoiceDate`, `Quantity`; optionally `Country` after transaction rules are agreed.
- **Why it matters:** A baseline shows demand concentration and time variation before comparing prices.
- **Method:** Aggregate quantity by product and week or month; use a small set of ranked product plots and a time-series or heatmap.
- **Could learn:** Which products account for observed volume and whether demand has visible seasonal or short-term patterns.
- **Limitations:** Quantity records purchases, not unmet demand; stock availability is absent, returns can distort totals, and the window is about one year.

### 2. Examine observed price-quantity association within products

- **Question:** For the same product, do transactions at different observed unit prices coincide with different quantities?
- **Variables:** `StockCode`, `UnitPrice`, `Quantity`, `InvoiceDate`, and optionally `Country`.
- **Why it matters:** This is a direct descriptive starting point for the pricing-demand objective.
- **Method:** First inspect within-product price variation; then use product-specific scatter/hexbin plots or quantity summaries across price bands for a limited set of products.
- **Could learn:** Whether there is enough price variation and a visible association to justify a more careful follow-up.
- **Limitations:** Association is not a causal price effect. Product mix, time, customer, promotions, bulk purchases, and stock constraints may influence both price and quantity; invoice-line quantities are not independent observations.

### 3. Compare demand across countries and customer activity

- **Question:** Is observed demand concentrated in particular countries or repeat customers?
- **Variables:** `Country`, `CustomerID`, `InvoiceNo`, `Quantity`, `UnitPrice` (with any sales value explicitly defined as a derived measure).
- **Why it matters:** Market/customer composition can explain aggregate patterns and guide which comparisons are meaningful.
- **Method:** Ranked summaries or Pareto plots by country and customer; compare invoice counts and quantities separately, and describe repeat activity only for identified customers.
- **Could learn:** Whether aggregate demand is driven by a few markets or high-activity customers.
- **Limitations:** Many rows lack `CustomerID`; a country is not necessarily the customer's residence; derived sales totals are sensitive to returns and the treatment of unusual prices.

**Suggested order:** confirm transaction rules before establishing demand patterns, then assess within-product price-quantity associations. Only the focused anomaly investigation has been performed; no rows were filtered, and the remaining proposals, pricing analysis, and broader EDA remain unperformed.

In [3]:
# Create analysis flags separately; the raw `retail` frame remains unchanged.
anomaly_flags = pd.DataFrame(
    {
        "negative_quantity": retail["Quantity"].lt(0),
        "invoice_starts_C": retail["InvoiceNo"].astype("string").str.startswith("C", na=False),
        "zero_unit_price": retail["UnitPrice"].eq(0),
        "negative_unit_price": retail["UnitPrice"].lt(0),
        "missing_customer_id": retail["CustomerID"].isna(),
        "missing_description": retail["Description"].isna(),
    },
    index=retail.index,
)

# A strict comparison group: positive quantity, positive price, and none of the six flags.
normal_mask = ~anomaly_flags.any(axis=1) & retail["Quantity"].gt(0)
normal_rows = retail.loc[normal_mask]

frequency = pd.DataFrame(
    {
        "rows": anomaly_flags.sum(),
        "share_of_all_rows_pct": anomaly_flags.mean().mul(100).round(5),
        "distinct_invoices": [
            retail.loc[mask, "InvoiceNo"].nunique() for mask in anomaly_flags.T.values
        ],
        "distinct_stock_codes": [
            retail.loc[mask, "StockCode"].nunique() for mask in anomaly_flags.T.values
        ],
    },
    index=anomaly_flags.columns,
)
print(f"Strict normal comparison rows: {len(normal_rows):,} ({len(normal_rows) / len(retail):.2%})")
print(f"Rows with at least one listed flag: {anomaly_flags.any(axis=1).sum():,} ({anomaly_flags.any(axis=1).mean():.2%})")
print("\nCategory frequency (flags overlap; percentages use all rows):")
display(frequency)

comparison_rows = []
comparison_groups = [("strict_normal", normal_mask)] + [
    (name, anomaly_flags[name]) for name in anomaly_flags.columns
]
for name, mask in comparison_groups:
    group = retail.loc[mask]
    comparison_rows.append(
        {
            "group": name,
            "rows": len(group),
            "quantity_median": group["Quantity"].median(),
            "quantity_q1": group["Quantity"].quantile(0.25),
            "quantity_q3": group["Quantity"].quantile(0.75),
            "unit_price_median": group["UnitPrice"].median(),
            "unit_price_q1": group["UnitPrice"].quantile(0.25),
            "unit_price_q3": group["UnitPrice"].quantile(0.75),
            "identified_customer_pct": group["CustomerID"].notna().mean() * 100,
            "date_start": group["InvoiceDate"].min(),
            "date_end": group["InvoiceDate"].max(),
        }
    )
print("\nCompact comparison with the strict normal group:")
comparison_table = pd.DataFrame(comparison_rows).set_index("group")
numeric_columns = comparison_table.select_dtypes(include="number").columns
display(comparison_table.assign(**comparison_table[numeric_columns].round(2)))

print("\nQuantity and indicative line-value totals:")
line_value = retail["Quantity"] * retail["UnitPrice"]
impact_summary = pd.DataFrame(
    {
        "row_count": [len(retail), int(retail["Quantity"].lt(0).sum()), int(anomaly_flags["invoice_starts_C"].sum()), len(normal_rows)],
        "quantity_sum": [retail["Quantity"].sum(), retail.loc[retail["Quantity"].lt(0), "Quantity"].sum(), retail.loc[anomaly_flags["invoice_starts_C"], "Quantity"].sum(), normal_rows["Quantity"].sum()],
        "quantity_times_price_sum": [line_value.sum(), line_value.loc[retail["Quantity"].lt(0)].sum(), line_value.loc[anomaly_flags["invoice_starts_C"]].sum(), (normal_rows["Quantity"] * normal_rows["UnitPrice"]).sum()],
    },
    index=["all_rows", "negative_quantity", "C_invoice", "strict_normal"],
)
display(impact_summary)
print("Line-value sums are descriptive Quantity × UnitPrice calculations, not verified accounting totals.")

print("\nNegative quantity by C-prefixed invoice (counts and row percentages):")
print(pd.crosstab(
    anomaly_flags["invoice_starts_C"],
    anomaly_flags["negative_quantity"],
    rownames=["invoice_starts_C"],
    colnames=["negative_quantity"],
    margins=True,
))
print("\nNegative-quantity rate within each invoice prefix group (%):")
print((pd.crosstab(
    anomaly_flags["invoice_starts_C"],
    anomaly_flags["negative_quantity"],
    normalize="index",
) * 100).round(2))

print("\nPairwise flag overlaps (non-zero intersections):")
overlap_rows = []
for left_index, left_name in enumerate(anomaly_flags.columns):
    for right_name in anomaly_flags.columns[left_index + 1:]:
        overlap_count = int((anomaly_flags[left_name] & anomaly_flags[right_name]).sum())
        if overlap_count:
            overlap_rows.append({"flag_1": left_name, "flag_2": right_name, "rows": overlap_count})
display(pd.DataFrame(overlap_rows).sort_values("rows", ascending=False).reset_index(drop=True))

print("\nRepresentative examples (fixed random seed; identifiers shown as stored):")
example_columns = [
    "InvoiceNo", "InvoiceDate", "StockCode", "Description", "Quantity", "UnitPrice", "CustomerID"
]
for name in anomaly_flags.columns:
    examples = retail.loc[anomaly_flags[name], example_columns]
    if not examples.empty:
        print(f"\n{name}")
        display(examples.sample(n=min(2, len(examples)), random_state=17))

Strict normal comparison rows: 397,884 (73.42%)


Rows with at least one listed flag: 144,025 (26.58%)

Category frequency (flags overlap; percentages use all rows):


,rows,share_of_all_rows_pct,distinct_invoices,distinct_stock_codes
negative_quantity,10624,1.96048,5172,2559
invoice_starts_C,9288,1.71394,3836,1947
zero_unit_price,2515,0.4641,2155,1419
negative_unit_price,2,0.00037,2,1
missing_customer_id,135080,24.92669,3710,3810
missing_description,1454,0.26831,1454,960



Compact comparison with the strict normal group:


,rows,quantity_median,quantity_q1,quantity_q3,unit_price_median,unit_price_q1,unit_price_q3,identified_customer_pct,date_start,date_end
group,,,,,,,,,,
strict_normal,397884,6.0,2.0,12.0,1.95,1.25,3.75,100.00,2010-12-01 08:26:00,2011-12-09 12:50:00
negative_quantity,10624,-2.0,-10.0,-1.0,2.10,1.06,4.95,83.82,2010-12-01 09:41:00,2011-12-09 11:58:00
invoice_starts_C,9288,-2.0,-6.0,-1.0,2.95,1.45,5.95,95.88,2010-12-01 09:41:00,2011-12-09 11:58:00
zero_unit_price,2515,-1.0,-32.0,3.0,0.00,0.00,0.00,1.59,2010-12-01 11:52:00,2011-12-08 15:24:00
negative_unit_price,2,1.0,1.0,1.0,-11062.06,-11062.06,-11062.06,0.00,2011-08-12 14:51:00,2011-08-12 14:52:00
missing_customer_id,135080,1.0,1.0,3.0,3.29,1.63,5.45,0.00,2010-12-01 11:52:00,2011-12-09 10:26:00
missing_description,1454,-3.0,-24.0,4.0,0.00,0.00,0.00,0.00,2010-12-01 11:52:00,2011-12-08 14:06:00


...

,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,CustomerID
299983,A563186,2011-08-12 14:51:00,B,Adjust bad debt,1,-11062.06,NaN
299984,A563187,2011-08-12 14:52:00,B,Adjust bad debt,1,-11062.06,NaN



missing_customer_id

,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,CustomerID
184026,552674,2011-05-10 15:07:00,48185,DOORMAT FAIRY CAKE,2,14.13,NaN
104815,545215,2011-02-28 16:47:00,22286,"DECORATION , WOBBLY RABBIT , METAL",2,3.29,NaN



missing_description


,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,CustomerID
435062,574073,2011-11-02 15:15:00,23579,NaN,3,0.0,NaN
74288,542418,2011-01-27 17:39:00,84966A,NaN,-17,0.0,NaN


## Focused anomaly investigation

**Scope and comparison.** The six flags below are non-exclusive: a row can belong to multiple categories, so their counts must not be added. The strict-normal comparison group (397,884 rows; 73.42%) has positive quantity and price and none of the six flags. All rates use 541,909 invoice lines as denominator. Flags were made in a separate frame; the raw `retail` data was not filtered or overwritten. Examples are small, fixed-seed samples, not evidence that every row in a category has the same meaning.

| Pattern | Rows (% of all rows) | Comparison and representative evidence | Interpretation, uncertainty, and later-analysis implication |
|---|---:|---|---|
| Negative `Quantity` | 10,624 (1.960%) | Median quantity -2 (IQR -10 to -1), versus strict-normal median 6 (IQR 2 to 12); median `UnitPrice` 2.10 versus 1.95. Spans the full observation period and 2,559 stock codes. Examples include C-prefixed invoice lines with identified customers and ordinary stock codes. | Negative item counts are consistent with returns or reversals, but signs alone do not establish the event or its link to an original sale. They sum to -484,531 units and -896,812.49 in `Quantity × UnitPrice`, so net-unit and line-value totals can differ materially from gross purchases. Keep as a distinct transaction class until business rules are confirmed. |
| `InvoiceNo` begins with `C` | 9,288 (1.714%) | Median quantity -2 (IQR -6 to -1), median price 2.95, and 95.88% have `CustomerID`; the dates span the full period. All 9,288 rows also have negative quantity. Of 10,624 negative-quantity rows, 1,336 do not have a C prefix; all 1,336 have zero price. | Prefix and sign form a strong cancellation/reversal-like pattern in this file, not proof of a specific business process. C-prefixed lines include both ordinary product codes and `M` / `Manual`, so not every line is necessarily a merchandise return. These rows account for -277,574 units and -896,812.49 in indicative line value; define treatment explicitly rather than counting them as ordinary purchases. |
| Zero `UnitPrice` | 2,515 (0.464%) | 2,475 (98.41%) lack `CustomerID`; all 1,454 missing-description rows are among the zero-price rows. Median quantity is -1 (IQR -32 to 3), and only 40 rows have a customer ID. Examples include description `smashed` with quantity -63 and an unidentified stock line with quantity -38. | Could include free/promotional items, write-offs, or other non-sale lines; negative quantities and missing fields make a blanket “free item” interpretation unsafe. These rows contribute units but zero under `Quantity × UnitPrice`, so they affect quantity-based measures and price summaries differently. Separate from paid transactions pending line-level classification. |
| Negative `UnitPrice` | 2 (0.00037%) | Both rows are dated 2011-08-12 one minute apart, `StockCode` B, `Description` “Adjust bad debt,” quantity +1, and missing customer. Each price is -11,062.06; strict-normal prices have median 1.95 (IQR 1.25 to 3.75). | The text and pairing suggest accounting adjustments rather than ordinary item prices, but source-system/accounting context is unavailable. Although only two rows, the simple quantity-times-price total is -22,124.12; they can distort average-price and revenue-like aggregates. Exclude from product unit-price analysis and keep separate for reconciliation, not as assumed data errors. |
| Missing `CustomerID` | 135,080 (24.927%) | By definition none have an identified customer; strict-normal rows all do. The missing-ID group has median quantity 1 (IQR 1 to 3), median price 3.29 (IQR 1.63 to 5.45), and spans nearly the full date range. It overlaps 1,719 negative-quantity rows, 2,475 zero-price rows, and both negative-price rows. | Many rows still look like ordinary positive-quantity, priced product lines; missing identity is not evidence that the line itself is invalid. Preserve for line/product/time summaries, but do not impute identity or use these rows as identified-customer activity. Customer-level results will cover a selected subset and may be compositionally different. |
| Missing `Description` | 1,454 (0.268%) | All 1,454 also have zero price and missing `CustomerID`; 862 have negative quantity. They span 960 stock codes. Examples show blank descriptions with both positive and negative quantities. | This concentration suggests the missing descriptions are associated with the same exceptional zero-price/unidentified lines, but does not reveal why the text is absent. The stock code remains available, so quantity-based product-code analysis may retain them; exclude from label-based interpretation until descriptions or code mappings are checked. |

**Relationships and impact.** The 9,288 C-prefixed rows are exactly the negative-quantity rows that have nonzero line value; the remaining 1,336 negative-quantity rows all have zero price. Thus the negative-quantity and C-prefix categories overlap heavily and must not be added as separate adjustment totals. Negative quantities total -484,531 units, while the strict-normal rows total 5,167,812 positive units. The full-file `Quantity × UnitPrice` sum is 9,747,747.934, but this arithmetic is not verified accounting revenue. Zero-price and missing-description lines can affect observed quantities and product mix without contributing to that line-value calculation. These are descriptive impacts only; they do not estimate causal effects or future demand.

**Provisional treatment recommendations**

| Category | Recommendation | Evidence-based basis | Assumption / unresolved point |
|---|---|---|---|
| Negative `Quantity` | `separate` | Negative signed units, with 87.4% on C-prefixed invoices; totals shown above. | Return/reversal interpretation is plausible, not verified; original sale linkage was not tested. |
| C-prefixed `InvoiceNo` | `separate` | Every flagged row has negative quantity. | Prefix meaning is inferred from this dataset’s pattern, not confirmed against system documentation. |
| Zero `UnitPrice` | `investigate further` | Strong overlap with missing customer/description and a mix of positive and negative quantities. | Whether individual rows are free goods, damage/write-off, or other events is unknown. |
| Negative `UnitPrice` | `separate` | Both rows are “Adjust bad debt” entries on code B, not ordinary item codes. | Accounting treatment needs confirmation before including them in financial totals. |
| Missing `CustomerID` | `keep` | Most such rows have positive quantities and positive prices; product and date fields remain available. | Keep only for analyses not requiring customer identity; do not infer or impute a customer. |
| Missing `Description` | `investigate further` | Every such row also has zero price and missing customer ID; `StockCode` remains present. | Description recovery/code validation is needed for name-based product claims. |

These are analysis-specific recommendations, not automatic cleaning rules. No rows were deleted, and no price-demand analysis or broader EDA was performed.

In [4]:
# Transaction structure: all rows are retained, including returns and unusual price lines.
print("TRANSACTION STRUCTURE (unfiltered raw rows)")
print(f"Rows: {len(retail):,}")
print(f"Distinct invoices: {retail['InvoiceNo'].nunique():,}")
print(f"Distinct stock codes: {retail['StockCode'].nunique():,}")
print(f"Rows with missing InvoiceNo / StockCode: {retail['InvoiceNo'].isna().sum():,} / {retail['StockCode'].isna().sum():,}")

invoice_stats = retail.groupby("InvoiceNo", dropna=False).agg(
    line_rows=("InvoiceNo", "size"),
    distinct_products=("StockCode", "nunique"),
)
multiple_line_invoices = invoice_stats["line_rows"].gt(1)
multiple_product_invoices = invoice_stats["distinct_products"].gt(1)
print("\nInvoice structure:")
print(f"Invoices with multiple rows: {multiple_line_invoices.sum():,} / {len(invoice_stats):,} ({multiple_line_invoices.mean():.1%})")
print(f"Invoices with multiple distinct products: {multiple_product_invoices.sum():,} / {len(invoice_stats):,} ({multiple_product_invoices.mean():.1%})")
print("Rows per invoice (median / 90th pct / 99th pct / max):", invoice_stats["line_rows"].quantile([0.5, 0.9, 0.99]).round(1).tolist(), int(invoice_stats["line_rows"].max()))
print("Distinct products per invoice (median / 90th pct / 99th pct / max):", invoice_stats["distinct_products"].quantile([0.5, 0.9, 0.99]).round(1).tolist(), int(invoice_stats["distinct_products"].max()))
print("Representative invoices with the most distinct products:")
display(invoice_stats.sort_values(["distinct_products", "line_rows"], ascending=False).head(5))

product_invoice_counts = retail.groupby("StockCode", dropna=False)["InvoiceNo"].nunique()
print("\nStockCode recurrence across invoices:")
print(f"Products observed in multiple invoices: {product_invoice_counts.gt(1).sum():,} / {len(product_invoice_counts):,} ({product_invoice_counts.gt(1).mean():.1%})")
print("Distinct invoices per product (median / 90th pct / 99th pct / max):", product_invoice_counts.quantile([0.5, 0.9, 0.99]).round(1).tolist(), int(product_invoice_counts.max()))
print("Five products appearing in the most invoices:")
display(product_invoice_counts.sort_values(ascending=False).head(5).rename("distinct_invoices").to_frame())

product_prices = retail.groupby("StockCode", dropna=False).agg(
    observations=("UnitPrice", "size"),
    distinct_prices=("UnitPrice", "nunique"),
    min_price=("UnitPrice", "min"),
    max_price=("UnitPrice", "max"),
)
variable_price_products = product_prices[product_prices["distinct_prices"].gt(1)]
print("\nObserved UnitPrice variation (exact distinct stored values; no price filtering):")
print(f"Products with multiple observed prices: {len(variable_price_products):,} / {len(product_prices):,} ({len(variable_price_products) / len(product_prices):.1%})")
print("Observations among variable-price products (median / 90th pct / max):", variable_price_products["observations"].quantile([0.5, 0.9]).round(1).tolist(), int(variable_price_products["observations"].max()))
print("Distinct prices among variable-price products (median / 90th pct / max):", variable_price_products["distinct_prices"].quantile([0.5, 0.9]).round(1).tolist(), int(variable_price_products["distinct_prices"].max()))
print("Five examples with the most observations:")
display(variable_price_products.sort_values("observations", ascending=False).head(5))

# Compare quantities at stored-row grain and after combining same product lines
# within an invoice and calendar date. The source dataframe is not changed.
line_groups = retail[["InvoiceNo", "StockCode", "InvoiceDate", "Quantity"]].copy()
line_groups["InvoiceDay"] = line_groups["InvoiceDate"].dt.normalize()
invoice_product_day = line_groups.groupby(
    ["InvoiceNo", "StockCode", "InvoiceDay"], dropna=False
).agg(
    Quantity=("Quantity", "sum"),
    source_rows=("Quantity", "size"),
)

def quantity_profile(values):
    return pd.Series({
        "observations": len(values),
        "mean": values.mean(),
        "median": values.median(),
        "q1": values.quantile(0.25),
        "q3": values.quantile(0.75),
        "negative_pct": values.lt(0).mean() * 100,
        "zero_pct": values.eq(0).mean() * 100,
        "positive_pct": values.gt(0).mean() * 100,
        "total_units": values.sum(),
    })

quantity_comparison = pd.DataFrame({
    "raw invoice line": quantity_profile(retail["Quantity"]),
    "invoice-product-day sum": quantity_profile(invoice_product_day["Quantity"]),
}).T
print("\nQuantity profile by observation grain:")
display(quantity_comparison.round(2))
collapsed_groups = invoice_product_day["source_rows"].gt(1)
print(f"Invoice-product-day groups combining multiple rows: {collapsed_groups.sum():,} ({collapsed_groups.mean():.2%} of groups)")
print(f"Total Quantity preserved by aggregation: {retail['Quantity'].sum():,} raw vs {invoice_product_day['Quantity'].sum():,} aggregated")
print("Examples of groups formed from repeated rows:")
display(invoice_product_day.loc[collapsed_groups].sort_values("source_rows", ascending=False).head(5))

TRANSACTION STRUCTURE (unfiltered raw rows)
Rows: 541,909


Distinct invoices: 25,900


Distinct stock codes: 4,070


Rows with missing InvoiceNo / StockCode: 0 / 0



Invoice structure:
Invoices with multiple rows: 20,059 / 25,900 (77.4%)
Invoices with multiple distinct products: 20,006 / 25,900 (77.2%)
Rows per invoice (median / 90th pct / 99th pct / max): [10.0, 47.0, 192.0] 1114
Distinct products per invoice (median / 90th pct / 99th pct / max): [10.0, 45.0, 191.0] 1110
Representative invoices with the most distinct products:


...

,observations,distinct_prices,min_price,max_price
StockCode,,,,
85123A,2313,8,0.00,5.91
22423,2203,10,0.00,32.04
85099B,2159,13,0.00,4.95
47566,1727,16,0.00,15.79
20725,1639,8,1.45,8.33



Quantity profile by observation grain:


,observations,mean,median,q1,q3,negative_pct,zero_pct,positive_pct,total_units
raw invoice line,541909.0,9.55,3.0,1.0,10.0,1.96,0.0,98.04,5176450.0
invoice-product-day sum,531225.0,9.74,3.0,1.0,10.0,1.97,0.0,98.03,5176450.0


Invoice-product-day groups combining multiple rows: 9,694 (1.82% of groups)
Total Quantity preserved by aggregation: 5,176,450 raw vs 5,176,450 aggregated
Examples of groups formed from repeated rows:


,,,Quantity,source_rows
InvoiceNo,StockCode,InvoiceDay,,
555524,22698,2011-06-05,20,20
C544580,S,2011-02-21,-16,16
555524,22697,2011-06-05,12,12
572861,22775,2011-10-26,96,8
572344,M,2011-10-24,744,7


## Transaction structure and unit of observation

All counts below use the unfiltered workbook, including negative-quantity, zero-price, and other flagged rows. The 541,909 rows contain 25,900 invoices and 4,070 stock codes; invoice and stock-code identifiers are complete. An invoice has a median of 10 rows and 10 distinct products (90th percentiles 47 and 45; maxima 1,114 rows and 1,110 products). In all, 20,059 invoices (77.4%) have multiple rows, and 20,006 (77.2%) have multiple distinct products. The largest examples are invoices 573585 (1,114 rows, 1,110 products), 581219 (749, 749), and 581492 (731, 731).

Stock codes recur across transactions: 3,837 of 4,070 (94.3%) appear in multiple invoices. The median code appears in 61 invoices; `85123A` is the most recurrent example (2,246 invoices). Exact stored `UnitPrice` varies for 3,603 products (88.5%). Among those products, the median has 80 rows and 4 distinct prices; the maxima are 2,313 rows and 687 prices. For example, `85123A` has 2,313 rows at 8 observed prices, ranging from 0.00 to 5.91; `23004` has 131 rows at 0.42 and 100 rows at 0.83. These are observed values, not necessarily comparable paid prices: zero/negative prices, returns, time, and transaction context remain mixed in.

At raw invoice-line grain, `Quantity` has median 3, interquartile range 1–10, and 1.96% negative values. Summing repeated rows by invoice, stock code, and calendar day creates 531,225 groups; 9,694 groups (1.82%) combine multiple source rows. Of those repeated groups, 528 (5.4%) have multiple observed prices, including cancellation-like invoice C544580 / code S with 16 rows at 15 prices. The aggregated median and quartiles remain 3 and 1–10, with 1.97% negative groups. The signed total remains 5,176,450, as expected from summation; it is not gross purchases because negative quantities offset positive ones. Examples include invoice 555524 / code 22698 (20 source rows sum to 20 units) and C544580 / code S (16 source rows sum to -16).

**A. True unit of observation.** A stored row is an invoice-line record for a stock code, with a quantity, observed unit price, and invoice timestamp. It is not an invoice, nor necessarily a unique product purchase: the same invoice/product/day can have multiple rows. The file has no separate line identifier in the listed columns, so the source row itself is the line record.

**B. Sensible future unit of analysis.** For product-price-demand relationships, a candidate is product × calendar period (for example, product-week), with units, transaction count, and a clearly defined price summary at that grain. This aligns a product's quantity and price over time and reduces repeated-line pseudo-replication. It is not yet a final choice: daily periods may be sparse, while wider periods can conceal short-lived price changes. Before selecting it, decide how to treat returns, zero/negative prices, invoice cancellations, and whether price should be transaction-weighted, quantity-weighted, or represented as a distribution. No demand or elasticity analysis is performed here.

**C. Information aggregation could lose.** Combining lines can hide within-period price variation, repeated entries, transaction-level quantity/basket context, exact timing, and distinctions between positive sales and negative reversals. Netting signed quantities can also make gross sales and returns invisible unless both are retained separately.

**D. Risks of staying at raw invoice-line level.** A large invoice can contribute hundreds of rows, and repeated records from one invoice are not independent demand events. Treating each line as a separate observation can overweight large baskets, inflate apparent sample size, and confound quantities with bulk purchases or line splitting. Raw rows also mix sales-like lines with returns, zero-price entries, and exceptional adjustments; their observed prices and quantities should not be treated as a homogeneous demand sample.

These counts describe the file's recorded structure. They do not establish why a line was recorded, why a price changed, or a causal price effect.

**Note:** computed on raw rows, including zero-price rows and non-product codes; recomputed on clean data in notebook 03.

In [5]:
# Price examples and a check for price differences hidden by invoice-product-day sums.
example_codes = (
    variable_price_products.loc[variable_price_products["distinct_prices"].eq(2)]
    .sort_values("observations", ascending=False)
    .head(3)
    .index
)
print("\nThree products with exactly two observed prices (price and row count):")
display(
    retail.loc[retail["StockCode"].isin(example_codes)]
    .groupby(["StockCode", "UnitPrice"])
    .size()
    .rename("observations")
    .reset_index()
)

priced_groups = retail.assign(InvoiceDay=retail["InvoiceDate"].dt.normalize()).groupby(
    ["InvoiceNo", "StockCode", "InvoiceDay"], dropna=False
).agg(
    source_rows=("Quantity", "size"),
    summed_quantity=("Quantity", "sum"),
    distinct_prices=("UnitPrice", "nunique"),
    min_price=("UnitPrice", "min"),
    max_price=("UnitPrice", "max"),
)
repeated_groups = priced_groups[priced_groups["source_rows"].gt(1)]
mixed_price_groups = repeated_groups[repeated_groups["distinct_prices"].gt(1)]
print("\nPrice variation inside repeated invoice-product-day groups:")
print(f"Repeated groups: {len(repeated_groups):,}; groups with multiple prices: {len(mixed_price_groups):,} ({len(mixed_price_groups) / len(repeated_groups):.1%} of repeated groups)")
print("Representative mixed-price groups:")
display(mixed_price_groups.sort_values(["source_rows", "distinct_prices"], ascending=False).head(5))


Three products with exactly two observed prices (price and row count):


,StockCode,UnitPrice,observations
0,23004,0.42,131
1,23004,0.83,100
2,23231,0.34,3
3,23231,0.42,257
4,23393,3.75,218
5,23393,10.79,24



Price variation inside repeated invoice-product-day groups:
Repeated groups: 9,694; groups with multiple prices: 528 (5.4% of repeated groups)
Representative mixed-price groups:


,,,source_rows,summed_quantity,distinct_prices,min_price,max_price
InvoiceNo,StockCode,InvoiceDay,,,,,
C544580,S,2011-02-21,16,-16,15,5.44,29.99
C544583,S,2011-02-21,7,-7,7,2.80,128.56
C553531,S,2011-05-17,7,-7,7,2.98,87.01
C558347,S,2011-06-28,6,-6,6,3.59,16.00
548813,M,2011-04-04,4,4,4,162.60,2382.92


In [6]:
# Compare candidate populations without changing or filtering the source dataframe.
restricted_mask = retail["Quantity"].gt(0) & retail["UnitPrice"].gt(0)
population_masks = {
    "broad_all_records": pd.Series(True, index=retail.index),
    "restricted_positive_quantity_positive_price": restricted_mask,
}
line_value = retail["Quantity"] * retail["UnitPrice"]

population_rows = []
for population_name, mask in population_masks.items():
    rows = retail.loc[mask]
    population_rows.append(
        {
            "population": population_name,
            "rows": len(rows),
            "pct_of_all_rows": len(rows) / len(retail) * 100,
            "distinct_invoices": rows["InvoiceNo"].nunique(),
            "distinct_products": rows["StockCode"].nunique(),
            "signed_quantity": rows["Quantity"].sum(),
            "quantity_times_unit_price": line_value.loc[mask].sum(),
        }
    )
population_summary = pd.DataFrame(population_rows).set_index("population")
print("Candidate population comparison (line value is Quantity × UnitPrice, not verified revenue):")
display(population_summary.round(2))

# These case flags intentionally overlap; their counts should not be summed.
case_masks = {
    "negative_quantity (return-like)": retail["Quantity"].lt(0),
    "InvoiceNo starts with C": retail["InvoiceNo"].astype("string").str.startswith("C", na=False),
    "zero UnitPrice": retail["UnitPrice"].eq(0),
    "negative UnitPrice (adjustment candidate)": retail["UnitPrice"].lt(0),
    "missing CustomerID": retail["CustomerID"].isna(),
    "missing Description": retail["Description"].isna(),
}
case_rows = []
for case_name, case_mask in case_masks.items():
    retained_restricted = int((case_mask & restricted_mask).sum())
    case_rows.append(
        {
            "case": case_name,
            "rows_in_broad": int(case_mask.sum()),
            "pct_of_all_rows": case_mask.mean() * 100,
            "rows_retained_restricted": retained_restricted,
            "pct_of_restricted_rows": retained_restricted / restricted_mask.sum() * 100,
        }
    )
case_summary = pd.DataFrame(case_rows).set_index("case")
print("\nImportant case flags (overlap is expected; broad retains all rows):")
display(case_summary.round(2))

# Exclusive descriptive labels use precedence only to make the class totals add up.
# The flags above preserve overlapping facts such as a zero-price return.
line_class = pd.Series("other", index=retail.index, dtype="string")
line_class.loc[retail["Quantity"].gt(0) & retail["UnitPrice"].gt(0)] = "sales_like"
line_class.loc[retail["Quantity"].lt(0)] = "return_like"
line_class.loc[retail["UnitPrice"].eq(0)] = "zero_price"
line_class.loc[retail["UnitPrice"].lt(0)] = "accounting_adjustment_candidate"
class_summary = pd.DataFrame(
    {
        "rows": line_class.value_counts(),
        "pct_of_all_rows": line_class.value_counts(normalize=True).mul(100),
        "signed_quantity": retail.groupby(line_class)["Quantity"].sum(),
        "quantity_times_unit_price": line_value.groupby(line_class).sum(),
    }
).fillna(0).sort_values("rows", ascending=False)
print("\nMutually exclusive descriptive classes (precedence: negative price, zero price, negative quantity, positive-positive):")
display(class_summary.round(2))

example_columns = ["InvoiceNo", "InvoiceDate", "StockCode", "Description", "Quantity", "UnitPrice", "CustomerID"]
example_masks = {
    "C-prefixed return-like rows": case_masks["InvoiceNo starts with C"],
    "zero-price rows": case_masks["zero UnitPrice"],
    "negative-price adjustment candidates": case_masks["negative UnitPrice (adjustment candidate)"],
    "restricted rows with missing CustomerID": restricted_mask & retail["CustomerID"].isna(),
    "missing-description rows": retail["Description"].isna(),
}
print("\nRepresentative rows (examples only; not classifications for every case):")
for example_name, example_mask in example_masks.items():
    print(f"\n{example_name}")
    display(retail.loc[example_mask, example_columns].head(2))

Candidate population comparison (line value is Quantity × UnitPrice, not verified revenue):


,rows,pct_of_all_rows,distinct_invoices,distinct_products,signed_quantity,quantity_times_unit_price
population,,,,,,
broad_all_records,541909,100.00,25900,4070,5176450,9747747.93
restricted_positive_quantity_positive_price,530104,97.82,19960,3922,5588376,10666684.54



Important case flags (overlap is expected; broad retains all rows):


,rows_in_broad,pct_of_all_rows,rows_retained_restricted,pct_of_restricted_rows
case,,,,
negative_quantity (return-like),10624,1.96,0,0.00
InvoiceNo starts with C,9288,1.71,0,0.00
zero UnitPrice,2515,0.46,0,0.00
negative UnitPrice (adjustment candidate),2,0.00,0,0.00
missing CustomerID,135080,24.93,132220,24.94
missing Description,1454,0.27,0,0.00



Mutually exclusive descriptive classes (precedence: negative price, zero price, negative quantity, positive-positive):


...

,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,CustomerID
299983,A563186,2011-08-12 14:51:00,B,Adjust bad debt,1,-11062.06,NaN
299984,A563187,2011-08-12 14:52:00,B,Adjust bad debt,1,-11062.06,NaN



restricted rows with missing CustomerID


,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,CustomerID
1443,536544,2010-12-01 14:32:00,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,2.51,NaN
1444,536544,2010-12-01 14:32:00,21774,DECORATIVE CATS BATHROOM BOTTLE,2,2.51,NaN



missing-description rows


,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,CustomerID
622,536414,2010-12-01 11:52:00,22139,NaN,56,0.0,NaN
1970,536545,2010-12-01 14:32:00,21134,NaN,1,0.0,NaN


## Candidate populations for initial price-demand analysis

The comparisons below are descriptive and use the original rows; neither population changes or cleans `retail`. The broad population retains every row and tags transaction types. The restricted population is defined mechanically as `Quantity > 0` and `UnitPrice > 0`. `Quantity × UnitPrice` is an arithmetic line-value total, not verified accounting revenue.

| Population | Rows retained (% of all) | Distinct invoices | Distinct products | Signed quantity | `Quantity × UnitPrice` total |
|---|---:|---:|---:|---:|---:|
| Broad: all records | 541,909 (100.00%) | 25,900 | 4,070 | 5,176,450 | 9,747,747.93 |
| Restricted: positive quantity and price | 530,104 (97.82%) | 19,960 | 3,922 | 5,588,376 | 10,666,684.54 |

The restricted rule excludes 11,805 rows (2.18%); this is a union, not the sum of overlapping case counts. All 10,624 negative-quantity rows, all 2,515 zero-price rows, and both negative-price rows are excluded. There are 1,336 rows that are both negative-quantity and zero-price, explaining much of the overlap. All 9,288 `C`-prefixed rows have negative quantity and are excluded; this is consistent with cancellation/return-like activity in this file, but the prefix's business meaning is not independently verified. The two negative-price rows are code `B`, described as “Adjust bad debt”; they are retained by broad and excluded by restricted. This text supports calling them adjustment candidates, not verified accounting treatment.

Missingness behaves differently from sign-based exclusions. Broad retains all 135,080 rows with missing `CustomerID`; restricted retains 132,220 (24.94% of restricted rows). Broad also retains all 1,454 missing-`Description` rows; restricted retains none because those rows have zero price. Missing customer identity therefore remains substantial under either population, whereas product-name coverage changes with the price rule.

For interpretation, the broad population can be separated into mutually exclusive descriptive classes using the precedence shown in the code: 530,104 positive-quantity/positive-price “sales-like” lines; 9,288 negative-quantity, nonzero-price “return-like” lines; 2,515 zero-price lines (including the 1,336 negative-quantity/zero-price rows); and 2 negative-price adjustment candidates. These labels describe signs and observed text/patterns, not confirmed business events. Their signed quantities are respectively 5,588,376, -277,574, -134,354, and +2; their line-value totals are 10,666,684.54, -896,812.49, 0, and -22,124.12.

**Broad population: trade-offs.** It preserves the full recorded transaction mix, including returns/reversals, zero-price activity, adjustment candidates, and missing identities. That is useful for coverage and sensitivity comparisons, and avoids defining the study sample solely by positive signs. Its limitation is heterogeneity: returns and adjustments are not necessarily demand at the recorded price, while net quantity and line-value totals combine different event types. Treating every broad row as a comparable price-demand observation could therefore distort the relationship.

**Restricted population: trade-offs.** Its sign rule yields a simple positive-quantity, positive-price sample that may better represent paid purchase lines for an initial descriptive association. But positive signs alone do not verify a completed sale or comparable transaction. It removes every return-like/cancellation-like, zero-price, and negative-price case; excludes all missing descriptions; loses 5,940 invoices and 148 products that have no qualifying line; and still retains substantial missing customer identity. Removing returns can bias results if returns are related to earlier prices, products, or demand; removing zero-price lines may omit promotions, gifts, or other activity, though their business meaning is unknown. Selection can change both product mix and observed price/quantity distributions.

**Why conclusions may differ.** Restricted totals are higher despite fewer rows because it excludes negative signed quantities and negative line values; zero-price lines add quantity but no line value. The two populations also represent different invoice and product coverage, not merely different sample sizes. An observed price-demand pattern can consequently shift because of transaction type and composition, even without any causal price effect. No final population is selected here; a later comparison should keep these classes visible and test conclusions under both definitions before deciding the analysis population.